## Step 1: Environment Setup

In [1]:
# Install libraries required
!pip install -q \
  transformers==4.41.2 \
  torchvision \
  pillow \
  bert-score \
  rouge-score \
  datasets

# Clone BARTScore repo and use it locally
!git clone https://github.com/neulab/BARTScore.git
%cd BARTScore

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.8 MB/s eta 0:00:00
ERROR: 

### Step 2: Load the Dataset

In [2]:
import pandas as pd

df = pd.read_csv("/kaggle/input/vqa-inference-dataset/Inference/combined_inference_vqa_single_answer.csv")
df['image_path'] = "/kaggle/input/vqa-inference-dataset/Inference/images/" + df['image_path']
df.head()

,image_path,question,answer
0,/kaggle/input/vqa-inference-dataset/Inference/...,What is the visible color of the chair?,Gray
1,/kaggle/input/vqa-inference-dataset/Inference/...,What is the average height of the back legs in...,4.125
2,/kaggle/input/vqa-inference-dataset/Inference/...,What kind of food is in the package?,Tortillas
3,/kaggle/input/vqa-inference-dataset/Inference/...,How many tortillas in total can the packaging ...,Six
4,/kaggle/input/vqa-inference-dataset/Inference/...,What is the display called?,Monitor


### Step 3: Load Pretrained VILT for VQA

In [3]:
from transformers import ViltProcessor, ViltForQuestionAnswering
from PIL import Image
import torch
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-finetuned-vqa")
model = ViltForQuestionAnswering.from_pretrained("dandelin/vilt-b32-finetuned-vqa").to(device)

2025-05-18 12:42:50.539084: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747572170.750157      18 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747572170.815997      18 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


preprocessor_config.json:   0%|          | 0.00/251 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/136k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/470M [00:00<?, ?B/s]

In [4]:
# Utility to print size
def print_model_size(model):
    import tempfile, os
    with tempfile.NamedTemporaryFile(delete=False) as f:
        torch.save(model.state_dict(), f.name)
        size = os.path.getsize(f.name) / 1e6
    print(f"Model size: {size:.2f} MB")

print_model_size(model)

Model size: 470.44 MB


In [5]:
# Check total and trainable parameters of the currently loaded VilBERT model
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 117,588,537
Trainable Parameters: 117,588,537


### Step 4: Run Inference

In [6]:
from PIL import Image
from tqdm import tqdm

predictions = []

for idx, row in tqdm(df.iterrows(), total = len(df)):
    try:
        image = Image.open(row['image_path']).convert("RGB")
        
        # Truncate and pad the question tokens to fit within model limits
        encoding = processor(
            images = image,
            text = row['question'],
            return_tensors = "pt",
            truncation = True,
            padding = "max_length",
            max_length = 40  # VilBERT's max token length
        ).to(device)

        outputs = model(**encoding)
        logits = outputs.logits
        predicted_idx = logits.argmax(-1).item()
        predicted_answer = model.config.id2label[predicted_idx]
        predictions.append(predicted_answer)

    except Exception as e:
        predictions.append("error")
        print(f"Error at index {idx}: {e}")

df["predicted_answer"] = predictions

100%|██████████| 2969/2969 [01:29<00:00, 33.19it/s]


In [7]:
# Check for non-string or NaN values
print(df["answer"].apply(type).value_counts())
print(df["predicted_answer"].apply(type).value_counts())

answer
<class 'str'>      2968
<class 'float'>       1
Name: count, dtype: int64
predicted_answer
<class 'str'>    2969
Name: count, dtype: int64


In [8]:
# Our df["answer"] column contains 3 float values
# Convert all answers to strings, replacing NaN (floats) with "missing"
df["answer"] = df["answer"].fillna("missing").astype(str)

### Step 5: Evaluate Metrics

#### Accuracy and F1

In [9]:
from sklearn.metrics import accuracy_score, f1_score

acc = accuracy_score(df["answer"], df["predicted_answer"])
f1 = f1_score(df["answer"], df["predicted_answer"], average = "macro")

print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")

Accuracy: 0.0000
F1 Score: 0.0000


#### BERTScore

In [10]:
from bert_score import score

P, R, F1 = score(df["predicted_answer"].tolist(), df["answer"].tolist(), lang = "en", verbose = True)
print(f"BERTScore (P): {P.mean():.4f}")
print(f"BERTScore (R): {R.mean():.4f}")
print(f"BERTScore (F1): {F1.mean():.4f}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/18 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/47 [00:00<?, ?it/s]

done in 1.33 seconds, 2231.45 sentences/sec
BERTScore (P): 0.9687
BERTScore (R): 0.9485
BERTScore (F1): 0.9578


#### ROUGE

In [11]:
from rouge_score import rouge_scorer

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_scores = [rouge.score(pred, gt)['rougeL'].fmeasure for pred, gt in zip(df['predicted_answer'], df['answer'])]
print(f"ROUGE-L: {sum(rouge_scores)/len(rouge_scores):.4f}")

ROUGE-L: 0.3257


#### BARTScore

In [12]:
from bart_score import BARTScorer

# Initialize the scorer
bart_scorer = BARTScorer(device = device, checkpoint = 'facebook/bart-large-cnn')

# Compute BARTScore
bart_scores = bart_scorer.score(df["predicted_answer"].tolist(), df["answer"].tolist(), batch_size=8)
print(f"BARTScore: {sum(bart_scores)/len(bart_scores):.4f}")

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BARTScore: -5.6399


### Step 6: Save Predictions and Metrics

In [13]:
df.to_csv("/kaggle/working/vqa_predictions.csv", index = False)